In [1]:
import pandas as pd
from google.colab import files

In [2]:
uploaded = files.upload()

# MUM01.20260416T170423.csv - Unemployment
# NAQ03.20260416T170401.csv - GDP
# TSM01.20260416T180411.csv - Exports
# ECBMRRFR.csv - Interest rate
# Inflation-data.xlsx - CPI

Saving ECBMRRFR.csv to ECBMRRFR.csv
Saving Inflation-data.xlsx to Inflation-data.xlsx
Saving MUM01.20260416T170423.csv to MUM01.20260416T170423.csv
Saving NAQ03.20260416T170401.csv to NAQ03.20260416T170401.csv
Saving TSM01.20260416T180411.csv to TSM01.20260416T180411.csv


In [3]:
unemp = pd.read_csv("MUM01.20260416T170423.csv")
gdp = pd.read_csv("NAQ03.20260416T170401.csv")
exports = pd.read_csv("TSM01.20260416T180411.csv")
rates = pd.read_csv("ECBMRRFR.csv")
cpi_raw = pd.read_excel("Inflation-data.xlsx", sheet_name="hcpi_q")

In [4]:
print("Unemployment shape:", unemp.shape)
print("GDP shape:", gdp.shape)
print("Exports shape:", exports.shape)
print("Rates shape:", rates.shape)
print("CPI raw shape:", cpi_raw.shape)

Unemployment shape: (339, 6)
GDP shape: (124, 5)
Exports shape: (674, 5)
Rates shape: (9968, 2)
CPI raw shape: (204, 227)


In [5]:
unemp = unemp[
    (unemp["Statistic Label"] == "Seasonally Adjusted Monthly Unemployment Rate") &
    (unemp["Age Group"] == "15 - 74 years") &
    (unemp["Sex"] == "Both sexes")].copy()

unemp["Month"] = pd.to_datetime(unemp["Month"], format="%Y %B")
unemp["Year"] = unemp["Month"].dt.year.astype(int)
unemp["Quarter"] = "Q" + unemp["Month"].dt.quarter.astype(str)

unemp_q = (
    unemp.groupby(["Year", "Quarter"], as_index=False)["VALUE"]
    .mean()
    .rename(columns={"VALUE": "Unemployment"})
)

print(f"Unemployment quarterly:")
print(unemp_q.head())

Unemployment quarterly:
   Year Quarter  Unemployment
0  1998      Q1      8.800000
1  1998      Q2      7.900000
2  1998      Q3      7.266667
3  1998      Q4      6.666667
4  1999      Q1      6.266667


In [6]:
gdp = gdp[gdp["Statistic Label"] == "GDP at Constant Market Prices"].copy()

gdp["Year"] = gdp["Quarter"].str[:4].astype(int)
gdp["Quarter"] = gdp["Quarter"].str[-2:]

gdp_q = gdp[["Year", "Quarter", "VALUE"]].rename(columns={"VALUE": "GDP"})

print(f"GDP quarterly:")
print(gdp_q.head())


GDP quarterly:
   Year Quarter    GDP
0  1995      Q1  26378
1  1995      Q2  27444
2  1995      Q3  27743
3  1995      Q4  27618
4  1996      Q1  28559


In [7]:
exports = exports[
    exports["Statistic Label"] == "Total Exports (Seasonally Adjusted)"
].copy()

exports["Month"] = pd.to_datetime(exports["Month"], format="%Y %B")
exports["Year"] = exports["Month"].dt.year.astype(int)
exports["Quarter"] = "Q" + exports["Month"].dt.quarter.astype(str)

exports_q = (
    exports.groupby(["Year", "Quarter"], as_index=False)["VALUE"]
    .mean()
    .rename(columns={"VALUE": "Exports"})
)

print(f"Exports quarterly:")
print(exports_q.head())

Exports quarterly:
   Year Quarter       Exports
0  1970      Q1  48671.666667
1  1970      Q2  47858.666667
2  1970      Q3  49144.000000
3  1970      Q4  52302.333333
4  1971      Q1  55638.333333


In [8]:
rates["observation_date"] = pd.to_datetime(rates["observation_date"])
rates["Year"] = rates["observation_date"].dt.year.astype(int)
rates["Quarter"] = "Q" + rates["observation_date"].dt.quarter.astype(str)

rates_q = (
    rates.groupby(["Year", "Quarter"], as_index=False)["ECBMRRFR"]
    .mean()
    .rename(columns={"ECBMRRFR": "InterestRate"})
)

rates_q["InterestRate"] = rates_q["InterestRate"].ffill()

print(f"Interest rates quarterly:")
print(rates_q.head())

Interest rates quarterly:
   Year Quarter  InterestRate
0  1999      Q1      3.000000
1  1999      Q2      2.543956
2  1999      Q3      2.500000
3  1999      Q4      2.809783
4  2000      Q1      3.197802


In [9]:
cpi_irl = cpi_raw[cpi_raw["Country Code"] == "IRL"].copy()

quarter_cols = [col for col in cpi_irl.columns if isinstance(col, int)]

cpi_long = cpi_irl.melt(
    id_vars=["Country Code", "Country"],
    value_vars=quarter_cols,
    var_name="Period",
    value_name="CPI"
)

cpi_long["Period"] = cpi_long["Period"].astype(str)
cpi_long["Year"] = cpi_long["Period"].str[:4].astype(int)
cpi_long["Quarter"] = "Q" + cpi_long["Period"].str[-1]

cpi_q = cpi_long[["Year", "Quarter", "CPI"]].copy()

print(f"CPI quarterly:")
print(cpi_q.head())

CPI quarterly:
   Year Quarter       CPI
0  1970      Q1  8.180063
1  1970      Q2  8.424414
2  1970      Q3  8.465170
3  1970      Q4  8.652523
4  1971      Q1  8.875320


In [10]:
df = unemp_q.merge(gdp_q, on=["Year", "Quarter"], how="inner")
df = df.merge(exports_q, on=["Year", "Quarter"], how="inner")
df = df.merge(rates_q, on=["Year", "Quarter"], how="inner")
df = df.merge(cpi_q, on=["Year", "Quarter"], how="inner")

df = df.sort_values(["Year", "Quarter"]).reset_index(drop=True)

In [11]:
print(f"Final dataset shape:", df.shape)
print(f"Missing values:")
print(df.isna().sum())

print(f"First 10 rows:")
print(df.head(10))

print(f"Last 10 rows:")
print(df.tail(10))

Final dataset shape: (104, 7)
Missing values:
Year            0
Quarter         0
Unemployment    0
GDP             0
Exports         0
InterestRate    0
CPI             0
dtype: int64
First 10 rows:
   Year Quarter  Unemployment    GDP       Exports  InterestRate       CPI
0  1999      Q1      6.266667  37412  5.030607e+06      3.000000  69.87321
1  1999      Q2      6.000000  38083  5.341812e+06      2.543956  70.75871
2  1999      Q3      5.433333  40112  5.832836e+06      2.500000  71.12095
3  1999      Q4      5.300000  40871  6.118408e+06      2.809783  71.84544
4  2000      Q1      4.800000  40867  6.108333e+06      3.197802  72.89193
5  2000      Q2      4.600000  42233  6.687900e+06      3.781250  74.46166
6  2000      Q3      4.233333  43730  7.118233e+06      3.781250  75.50815
7  2000      Q4      3.900000  44364  8.000500e+06      3.781250  76.55464
8  2001      Q1      3.966667  43987  7.824500e+06      3.781250  76.75589
9  2001      Q2      4.000000  44926  7.547933e+06

In [12]:
df = df[["Year", "Quarter", "Unemployment", "GDP", "CPI", "InterestRate", "Exports"]]

In [13]:
df.to_csv("final_macro_dataset.csv", index=False)
files.download("final_macro_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
df["Quarter_num"] = df["Quarter"].str.replace("Q", "").astype(int)

df["Date"] = pd.PeriodIndex(year=df["Year"], quarter=df["Quarter_num"]).to_timestamp()

df = df.sort_values("Date").reset_index(drop=True)

df = df.drop(columns=["Quarter_num"])

/tmp/ipykernel_17286/1630132768.py:3: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  df["Date"] = pd.PeriodIndex(year=df["Year"], quarter=df["Quarter_num"]).to_timestamp()


In [15]:
df.head()

,Year,Quarter,Unemployment,GDP,CPI,InterestRate,Exports,Date
0,1999,Q1,6.266667,37412,69.87321,3.000000,5.030607e+06,1999-01-01
1,1999,Q2,6.000000,38083,70.75871,2.543956,5.341812e+06,1999-04-01
2,1999,Q3,5.433333,40112,71.12095,2.500000,5.832836e+06,1999-07-01
3,1999,Q4,5.300000,40871,71.84544,2.809783,6.118408e+06,1999-10-01
4,2000,Q1,4.800000,40867,72.89193,3.197802,6.108333e+06,2000-01-01


In [16]:
df = df[["Year", "Quarter", "Date", "Unemployment", "GDP", "CPI", "InterestRate", "Exports"]]

In [17]:
df.to_csv("final_macro_dataset_date.csv", index=False)
files.download("final_macro_dataset_date.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>